# Lab 2 — Regulomes and hypergraphs

*Second session of the [`notebooks/` course](README.md) on computational synthetic morphology.*

A **gene regulatory network** (GRN) is the wiring diagram of development: which transcription
factors (TFs) switch which target genes on or off, in which cells, when. A **regulome** is that
wiring inferred genome-wide from data — for us, the cerebral-organoid regulome of **Fleck et al.
(2023, *Nature*)**, reconstructed from single-cell multiome (RNA + ATAC) with the Pando method:
**720 TFs**, **2,792 genes**, ~**75k** TF→gene edges.

The central modelling decision of this whole project is *how to represent that regulome*. A
regulon — one TF plus the set of genes it regulates — is a **set with a shared cause**, not a
clique of genes. The natural object for "a shared cause linking many elements" is a **hyperedge**;
the natural object for a collection of them is a **hypergraph**. A pairwise graph throws the
shared-cause structure away. This lab makes that concrete:

1. What a regulome is, and what the Fleck incidence matrix looks like.
2. Why a hypergraph, not a graph — the clique-expansion blow-up *and* aliasing.
3. Hypergraphs in `hgx`: the incidence matrix *is* the hypergraph; node/edge degrees; the
   bipartite (star) and clique expansions; one hypergraph-convolution forward pass.
4. A first peek at structure — the hypergraph Laplacian and its spectral gap (the seed of the
   *Module Identifiability Index* you'll meet in Lab 4).
5. Exercises.

**Needs:** `hgx`, `numpy`, `matplotlib` (a `uv sync` of this repo gives you all three). The Fleck
regulome is committed under `data/processed/`; if it is missing the notebook falls back to a tiny
synthetic regulome so every cell still runs.

In [ ]:
import json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt

import jax
import jax.numpy as jnp
import hgx

rng = np.random.default_rng(0)

# --- locate the repo's data/processed/ (robust to where the notebook is launched) ---
def _find(path_parts):
    here = Path.cwd()
    for base in [here, *here.parents]:
        p = base.joinpath(*path_parts)
        if p.exists():
            return p
    return None

PROC = _find(["data", "processed"])
if PROC is not None and (PROC / "incidence.npy").exists():
    incidence  = np.load(PROC / "incidence.npy").astype(np.float32)        # (genes, regulons)
    gene_names = json.loads((PROC / "gene_names.json").read_text())
    tf_names   = json.loads((PROC / "tf_names.json").read_text())          # one name per regulon
    key_tf     = json.loads((PROC / "key_tf_indices.json").read_text())    # {TF: regulon_index}
    module_labels = (np.load(PROC / "module_labels.npy")
                     if (PROC / "module_labels.npy").exists() else None)
    pca_features  = (np.load(PROC / "node_features_pca.npy").astype(np.float32)
                     if (PROC / "node_features_pca.npy").exists() else None)
    SOURCE = "Fleck et al. 2023 organoid regulome (Pando)"
else:
    print("[note] data/processed/ not found — using a tiny synthetic regulome so the lab still runs.")
    print("       Run `python scripts/00_preprocess.py` (with the Zenodo data) for the real Fleck regulome.")
    # 12 genes, 4 regulons (3 of them share genes on purpose)
    gene_names = [f"g{i}" for i in range(12)]
    tf_names   = ["TF_A", "TF_B", "TF_C", "TF_D"]
    regulons   = [[0,1,2,3,4], [3,4,5,6,7], [6,7,8,9], [1,2,8,10,11]]
    incidence  = np.zeros((12, 4), dtype=np.float32)
    for j, mem in enumerate(regulons):
        for g in mem:
            incidence[g, j] = 1.0
    key_tf = {"TF_A": 0, "TF_B": 1}
    module_labels = incidence.argmax(1).astype(int)
    pca_features  = rng.normal(size=(12, 8)).astype(np.float32)
    SOURCE = "synthetic toy regulome (12 genes, 4 regulons)"

n_genes, n_regulons = incidence.shape
print(f"loaded: {SOURCE}")
print(f"  genes (nodes):   {n_genes}")
print(f"  regulons (edges):{n_regulons}")
print(f"  TF→gene incidences (nonzeros): {int(incidence.sum()):,}")

## 1. What is a regulome?

In Davidson's framework, development is run by **GRN kernels** — small, deeply conserved,
recursively-wired subcircuits that *define a body part* — wrapped in plug-ins, switches, and
differentiation gene batteries. A single-cell **regulome** is the data-driven version of that
diagram: for each TF, the set of genes it (directly or indirectly) regulates, the *regulon*.

For us the substrate is the **Fleck et al. (2023) cerebral-organoid regulome**, inferred with
**Pando** from multiome data (chromatin accessibility tells you where a TF *could* bind; RNA tells
you what changes when it does). It comes to us as a binary **incidence matrix** $H \in
\{0,1\}^{\text{genes} \times \text{regulons}}$: $H_{g,r} = 1$ iff gene $g$ is a member of regulon
$r$ (regulon $r$ being "TF $r$ and its targets"). The single TF heading a regulon is itself a
member of its own regulon (TFs regulate other TFs — that recursion is the point).

The thing to internalise: **a regulon is a set with a shared cause** (the TF), not a clique of
genes that happen to be pairwise related. We will see in a moment why keeping that distinction
matters.

In [ ]:
# regulon-size distribution (how many genes each TF regulates)
reg_size = incidence.sum(axis=0)
# gene-membership distribution (in how many regulons does each gene appear)
gene_mem = incidence.sum(axis=1)

print("regulon size  — min %d  median %d  mean %.1f  max %d"
      % (reg_size.min(), np.median(reg_size), reg_size.mean(), reg_size.max()))
print("gene membership — min %d  median %d  mean %.1f  max %d"
      % (gene_mem.min(), np.median(gene_mem), gene_mem.mean(), gene_mem.max()))
density = incidence.mean()
print(f"incidence density: {density:.3%}  (a regulon touches ~{density*n_genes:.0f} of {n_genes} genes on average)")

fig, ax = plt.subplots(1, 3, figsize=(15, 4))
ax[0].hist(reg_size, bins=min(40, int(reg_size.max())), color="#4393C3")
ax[0].set_title("regulon sizes  (genes per TF)"); ax[0].set_xlabel("size"); ax[0].set_ylabel("# regulons")
ax[1].hist(gene_mem, bins=min(40, int(gene_mem.max())), color="#D6604D")
ax[1].set_title("gene memberships  (regulons per gene)"); ax[1].set_xlabel("# regulons"); ax[1].set_ylabel("# genes")
# a corner of the incidence matrix, sorted by regulon size and gene membership for legibility
gi = np.argsort(-gene_mem)[:80]; ri = np.argsort(-reg_size)[:60]
ax[2].imshow(incidence[np.ix_(gi, ri)], aspect="auto", cmap="Greys", interpolation="nearest")
ax[2].set_title("incidence H  (top-80 genes × top-60 regulons)"); ax[2].set_xlabel("regulon"); ax[2].set_ylabel("gene")
plt.tight_layout(); plt.show()

## 2. Why a hypergraph, not a graph?

The tempting shortcut is to turn each regulon into a **clique**: if genes $g_1,\dots,g_k$ are all
in regulon $r$, draw an edge between every pair. This **clique expansion** gives you an ordinary
graph you can throw standard tools at — but it is *lossy* in two ways:

1. **Blow-up.** A regulon of $k$ genes becomes $\binom{k}{2}$ edges. With regulons of mean size
   ~60, the Fleck regulome's ~75k incidences explode into **millions** of pairwise edges — most of
   them spurious in the sense that they were never *separately* asserted.
2. **Aliasing (the real problem).** From the clique-expansion graph alone you **cannot recover the
   regulon structure**. Two genes connected by an edge might be co-regulated by *one* TF, or by two
   *different* TFs that happen to share a target. Distinct hypergraphs collapse to the *same* graph.
   The "shared cause" — which TF, and which other genes share that cause — is exactly the
   higher-order information a hyperedge keeps and a pairwise edge destroys.

Everything downstream in this project — hypergraph neural networks (Lab 5), the Hodge-Laplacian
*Module Identifiability Index* (Lab 4), the regulon-overlap *fidelity score* (Lab 3) — operates on
the **incidence matrix $H$ directly**, never on a clique expansion, precisely so it can see that
structure. Let's watch the information disappear.

In [ ]:
# --- a tiny, hand-built example: two regulons sharing one gene ---
# regulon X = {gA, gB, gC},  regulon Y = {gC, gD, gE}   (gC is the shared target)
toy_genes = ["gA", "gB", "gC", "gD", "gE"]
H_xy = np.array([
    # X  Y
    [1, 0],   # gA
    [1, 0],   # gB
    [1, 1],   # gC  (in both)
    [0, 1],   # gD
    [0, 1],   # gE
], dtype=np.float32)
hg_xy   = hgx.from_incidence(jnp.asarray(H_xy))
A_xy    = np.asarray(hgx.clique_expansion(hg_xy))            # the pairwise graph

# --- a DIFFERENT pair of regulons that produces the SAME clique-expansion graph ---
# regulon P = {gA, gB, gC, gD, gE} minus the pair (gA,gD),(gA,gE),(gB,gD),(gB,gE)? no — simplest:
# P = {gA,gB,gC},  Q = {gC,gD,gE}  is the same as X,Y.  To alias, use:
# P = {gA,gB,gC},  Q = {gC,gD,gE},  R = {} ... -> trivially same.  The honest demo:
# {gA,gB,gC,gD,gE} as ONE regulon  vs  {gA,gB,gC}+{gC,gD,gE}  give DIFFERENT graphs (the first is K5),
# but {gA,gB,gC}+{gC,gD}+{gC,gE} (three regulons) gives a graph where gC—gD and gC—gE but no gD—gE,
# i.e. NOT the same as X,Y either.  The clean aliasing example: swap which gene is shared.
H_pq = np.array([
    # P  Q
    [1, 0],   # gA
    [1, 1],   # gB  (now gB is the shared one)
    [1, 0],   # gC
    [0, 1],   # gD
    [0, 1],   # gE
], dtype=np.float32)
hg_pq = hgx.from_incidence(jnp.asarray(H_pq))
A_pq  = np.asarray(hgx.clique_expansion(hg_pq))

def show_graph(ax, A, names, title):
    A = (A > 0).astype(int); k = len(names)
    th = np.linspace(0, 2*np.pi, k, endpoint=False); xy = np.c_[np.cos(th), np.sin(th)]
    for i in range(k):
        for j in range(i+1, k):
            if A[i, j]:
                ax.plot(xy[[i,j],0], xy[[i,j],1], color="#888", lw=1.5, zorder=1)
    ax.scatter(xy[:,0], xy[:,1], s=400, color="#4393C3", zorder=2)
    for i, nm in enumerate(names):
        ax.text(xy[i,0], xy[i,1], nm, ha="center", va="center", zorder=3, fontsize=9)
    ax.set_title(title); ax.set_xlim(-1.4, 1.4); ax.set_ylim(-1.4, 1.4); ax.set_aspect("equal"); ax.axis("off")

print("H for {X={gA,gB,gC}, Y={gC,gD,gE}}:\n", H_xy.astype(int), "\n")
print("H for {P={gA,gB,gC}, Q={gC,gD,gE}} but with gB as the shared gene:\n", H_pq.astype(int), "\n")
print("These two hypergraphs are clearly different (different regulons).")
print("Their clique-expansion graphs differ here — good.  But note how much you've already lost:")
print("from the graph alone you can't tell that gA,gB,gC formed *one* regulon vs three pairwise facts.")

fig, ax = plt.subplots(1, 2, figsize=(11, 4.5))
show_graph(ax[0], A_xy, toy_genes, "clique expansion of {X, Y}")
show_graph(ax[1], A_pq, toy_genes, "clique expansion of {P, Q} (gB shared)")
plt.tight_layout(); plt.show()

# --- the blow-up, on the real regulome ---
hg_full = hgx.from_incidence(jnp.asarray(incidence))
A_full  = np.asarray(hgx.clique_expansion(hg_full))
n_clique_edges = int((A_full > 0).sum() // 2)
n_incidences   = int(incidence.sum())
print(f"\nFull '{SOURCE}':")
print(f"  TF→gene incidences in H : {n_incidences:,}")
print(f"  edges after clique expansion : {n_clique_edges:,}   ({n_clique_edges / max(n_incidences,1):.0f}× more)")
print("  ...and you still can't read the regulons back off that graph.")

## 3. Hypergraphs in `hgx`

`hgx` keeps the hypergraph *as the incidence matrix* — no clique expansion. `hgx.from_incidence(H)`
gives you a `Hypergraph` whose `.incidence` is exactly $H$. Two basic quantities:

- **`node_degrees`** — for each gene, the number of regulons it belongs to (the column sum of its
  row of $H$): "how broadly co-regulated is this gene?"
- **`edge_degrees`** — for each regulon, the number of genes in it (the regulon size).

Two *lossless* views are available when you want a graph-shaped object: `hg.star_expansion()` — the
bipartite **gene ↔ regulon** incidence graph (a regulon becomes a single hub node connected to its
genes; nothing is lost, the hypergraph is recoverable) — and `hgx.clique_expansion(hg)` — the
**lossy** pairwise graph from §2.

And the reason to keep the hypergraph: **message passing respects the shared cause**. A hypergraph
convolution does two aggregations — *gene → regulon* (pool the genes of each regulon) then *regulon
→ gene* (a gene gathers from the regulons it's in). One `hgx.UniGCNConv` layer is exactly that. We
run a forward pass below, mapping the 97-dimensional PCA features that summarise each gene's
expression into a 16-dimensional learned embedding.

In [ ]:
hg = hgx.from_incidence(jnp.asarray(incidence),
                        node_features=(jnp.asarray(pca_features) if pca_features is not None else None))
print(f"hg: {hg.num_nodes} genes (nodes), {hg.num_edges} regulons (hyperedges)")

node_deg = np.asarray(hg.node_degrees)     # regulons per gene
edge_deg = np.asarray(hg.edge_degrees)     # genes per regulon

# the most broadly co-regulated genes
top_genes = np.argsort(-node_deg)[:10]
print("\nmost broadly co-regulated genes (in the most regulons):")
for i in top_genes:
    print(f"  {gene_names[i]:<12s}  in {int(node_deg[i])} regulons")

# the biggest regulons (the broadest regulators)
top_regs = np.argsort(-edge_deg)[:10]
print("\nbiggest regulons (broadest regulators):")
for j in top_regs:
    print(f"  {tf_names[j]:<12s}  regulates {int(edge_deg[j])} genes")

# a hypergraph-convolution forward pass: 97-D PCA features -> 16-D embedding
if pca_features is not None:
    conv = hgx.UniGCNConv(pca_features.shape[1], 16, key=jax.random.PRNGKey(0))
    emb  = np.asarray(conv(hg))                          # (n_genes, 16)
    print(f"\nUniGCNConv forward pass: {pca_features.shape} --(gene→regulon→gene)--> {emb.shape}")
    print(f"  embedding row-norm: mean {np.linalg.norm(emb,axis=1).mean():.3f}  std {np.linalg.norm(emb,axis=1).std():.3f}")
    print("  (a single untrained layer already mixes each gene with its regulon-mates — this is the")
    print("   message-passing primitive every model in Labs 3–5 is built from.)")
else:
    print("\n(no PCA features in the toy fallback — skipping the UniGCNConv forward pass)")

## 4. A first peek at structure: the hypergraph Laplacian

Just as a graph has a Laplacian whose spectrum tells you about connectivity and communities, a
hypergraph has one too. `hgx.hypergraph_laplacian(hg, normalized=True)` returns the symmetric
normalised vertex Laplacian
$$
L \;=\; I \;-\; D_v^{-1/2}\, H\, W\, D_e^{-1}\, H^{\top}\, D_v^{-1/2},
$$
where $D_v$ is the diagonal of node degrees, $D_e$ of edge degrees, $W$ the (here, identity)
hyperedge weights. Its eigenvalues live in $[0, \le 1]$. The smallest is (essentially) zero — the
harmonic component, one per connected piece of the regulome. The **gap above it** — how big
$\lambda_2$ is, and how the low eigenvalues are spaced — measures how cleanly the regulome
*separates into modules*: a large gap ⇒ a few well-separated regulatory blocks; eigenvalues
crowded near zero ⇒ a diffuse, hard-to-decompose tangle.

That single number is the seed of the **Module Identifiability Index** you'll build in Lab 4 — and
the full higher-order story is `hgx.hodge_laplacians(hg, max_dim=2)` (the chain of Laplacians
$L_0, L_1, \dots$, of which $L_0$ is essentially this one). For now, just look at the spectrum.

In [ ]:
L = np.asarray(hgx.hypergraph_laplacian(hg, normalized=True))
ev = np.linalg.eigvalsh(L)                                  # ascending
ev = np.clip(ev, 0.0, None)                                 # tiny negatives -> 0 (numerical)
n_zero = int((ev < 1e-6).sum())
lam2 = ev[n_zero] if n_zero < len(ev) else float("nan")     # algebraic connectivity (Fiedler value)
gap  = (ev[min(n_zero+10, len(ev)-1)] - ev[n_zero]) if n_zero < len(ev) else float("nan")

print(f"hypergraph Laplacian: {L.shape},  symmetric={np.allclose(L, L.T, atol=1e-4)}")
print(f"  ~zero eigenvalues (harmonic / connected pieces): {n_zero}")
print(f"  algebraic connectivity  λ₂ ≈ {lam2:.4f}")
print(f"  spread over the next 10 nontrivial eigenvalues ≈ {gap:.4f}   (bigger ⇒ sharper modules)")

fig, ax = plt.subplots(1, 2, figsize=(13, 4))
ax[0].plot(ev, ".", ms=4); ax[0].axhline(0, color="k", lw=0.5)
ax[0].set_title("hypergraph-Laplacian spectrum (sorted)"); ax[0].set_xlabel("index"); ax[0].set_ylabel("eigenvalue")
# a 2-D spectral embedding from the first two nontrivial eigenvectors, coloured by regulon-of-max-membership
w, V = np.linalg.eigh(L)
order = np.argsort(w)
fiedler = V[:, order[n_zero:n_zero+2]] if n_zero+1 < len(order) else V[:, :2]
c = (module_labels if module_labels is not None else incidence.argmax(1))
sc = ax[1].scatter(fiedler[:,0], fiedler[:,1], c=c, cmap="tab20", s=8, alpha=0.7)
ax[1].set_title("spectral embedding of genes\n(first 2 nontrivial Laplacian eigenvectors; colour = a regulon label)")
ax[1].set_xlabel("eigvec 1"); ax[1].set_ylabel("eigvec 2")
plt.tight_layout(); plt.show()
print("genes from the same regulon land near each other — the hypergraph structure is *spatially* visible.")

## 5. Exercises

**(a) Regulon overlap — the higher-order signal a graph blurs.**
Pick a master TF (e.g. `GLI3`, `FOXG1`, `TBR1`, `DLX1`, `DLX2`, `EOMES`, `NEUROD6` — all of
which head a regulon; note that `key_tf_indices.json` actually maps each name to its *gene*
row index, while the *regulon* column of $H$ is `tf_names.index(name)`). List its regulon's
target genes (the rows of $H$ that are 1 in its column). Then pick a second
master TF and compute the **Jaccard overlap** of their two regulons,
$|R_1 \cap R_2| / |R_1 \cup R_2|$, and print the shared genes. (This overlap — "which genes share a
regulator with which others" — is exactly what the *fidelity score* in Lab 3 uses, and exactly what
the clique-expansion graph cannot tell you.) Which pair of master TFs overlaps most? Least? Does it
match what you'd expect from their biology (e.g. `DLX1`/`DLX2` are paralogous interneuron TFs)?

**(b) Is the regulon-size distribution heavy-tailed?**
Plot `reg_size` on log–log axes (rank vs size, or a log-binned histogram). Real GRNs tend to be
scale-free-ish — a few hub TFs with huge regulons, many with small ones. Estimate the tail slope.

**(c) Graph degree vs hypergraph degree.**
For a handful of genes, compare the gene's degree in the **clique-expansion graph** `A_full` (number
of distinct pairwise neighbours) with its **hypergraph node degree** `node_deg` (number of regulons
it's in). When are these wildly different, and what does a gene with *low* hypergraph degree but
*high* graph degree tell you? (Hint: it's in one or two *big* regulons.)

A starter for (a):

In [ ]:
# --- Exercise (a) starter ---
# Note: key_tf_indices.json maps a TF name to its *gene* row index (its position in the
# expression matrix); the *regulon* (column of H) headed by a TF is found via tf_names.
def regulon_column(tf_name):
    try:
        return tf_names.index(tf_name)
    except ValueError:
        raise KeyError(f"{tf_name!r} does not head a regulon; choices include: {list(key_tf)}")

def regulon_targets(tf_name):
    j = regulon_column(tf_name)
    return set(gene_names[i] for i in np.where(incidence[:, j] > 0)[0])

def jaccard(a, b):
    a, b = set(a), set(b)
    return len(a & b) / max(len(a | b), 1)

available = [t for t in key_tf if t in tf_names]          # master TFs that head a regulon
print("available master TFs:", available)
if len(available) >= 2:
    t1, t2 = available[0], available[1]
    R1, R2 = regulon_targets(t1), regulon_targets(t2)
    print(f"\n{t1}: regulon of {len(R1)} genes  (column {regulon_column(t1)} of H)")
    print(f"{t2}: regulon of {len(R2)} genes  (column {regulon_column(t2)} of H)")
    print(f"Jaccard({t1}, {t2}) = {jaccard(R1, R2):.3f}")
    print(f"shared genes ({len(R1 & R2)}): {sorted(R1 & R2)[:20]}{' ...' if len(R1 & R2) > 20 else ''}")

    # all-pairs overlap among the master TFs
    print("\nall-pairs Jaccard overlap among master TFs:")
    avail2 = [(t, regulon_targets(t)) for t in available]
    for i in range(len(avail2)):
        for k in range(i + 1, len(avail2)):
            (na, Ra), (nb, Rb) = avail2[i], avail2[k]
            print(f"  {na:<9s} ∩ {nb:<9s}  J = {jaccard(Ra, Rb):.3f}   ({len(Ra & Rb)} shared)")

# now you write (b) and (c).  Hints: np.histogram with log-spaced bins for (b);
# (A_full > 0).sum(axis=1) vs node_deg for (c).

## Recap & next

You now have the project's core data object in your hands: the regulome as a **hypergraph**, stored
as an incidence matrix $H$, with `hgx` operating on $H$ directly because the alternative — a
clique-expansion graph — would blow up *and* blur the "which TF, which co-regulated genes" structure
that everything downstream needs. You've seen the basic operations (`from_incidence`, `node_degrees`
/ `edge_degrees`, `star_expansion`, `clique_expansion`), one hypergraph-convolution forward pass
(the message-passing primitive), and a first structural readout (the Laplacian spectrum, whose gap
foreshadows the *Module Identifiability Index*).

**Lab 3 — Benchmarking fidelity:** does this organoid regulome actually *predict* what happens when
you knock a TF down in primary human cortex? Regulon overlap, perturbation-direction concordance,
cross-species conservation. (Methods §2.2, §2.6 and Results §3.1–3.3 of `publication/paper.Rnw`.)

*References:* Davidson, *The Regulatory Genome* (2006); Fleck JS et al., *Nature* 621:365–372 (2023);
Feng et al., *AAAI* 2019 (hypergraph neural networks); the §1.4 / §2.2 material of the whitepaper.